In [10]:
from dotenv import load_dotenv
load_dotenv()
from gitsource import chunk_documents
import requests
import io
import zipfile
import frontmatter
from pprint import pprint
from gitsource import chunk_documents
from minsearch import Index 
import json
from openai import OpenAI

#utilites
def llm(user_prompt, instructions=None, model='gpt-4o-mini'):
    messages=[]

    if instructions is not None:
        messages.append({
            "role":"system",
            "content":instructions
        })
    messages.append({
        "role":"user",
        "content":user_prompt
    })
    openai_client=OpenAI()
    response=openai_client.responses.create(
        model=model,
        input=messages
    )
    return response.output_text



def rag(query):
    search_results=search(query)
    prompt=build_prompt(query,search_result)
    answer= llm(prompt, instructions)

def search(query):
    return chunk_index.search(query, num_results=5)

def build_prompt(query,search_result):
    search_result_json=json.dumps(search_result,indent=2)
    user_prompt = f"""
    <QUESTION>
    {query}
    </QUESTION>

    <CONTEXT>
    {search_result_json}
    </CONTEXT>
    """.strip()

    return user_prompt


query="LLM as a judge"


#get documents 
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)

files = reader.read()

print(f"Loaded {len(files)} documents")
documents = [f.parse() for f in files]

#chunking
document_chunks = []
document_chunks = chunk_documents(documents, size=3000, step=1500)

chunk_index=Index(
    text_fields=["title","description","content"],
    keyword_fields=["filename"]
)
chunk_index.fit(document_chunks)


#do search
search_result=chunk_index.search(query, num_results=5)
query='how do i implement llm as judge?'
search_result_json=json.dumps(search_result,indent=2)
#print(search_result_json)

#build prompt

instructions="""
you re a course assistant, your task is to answer the QUESTION from the course students using the provided CONTEXT
"""

user_prompt=f"""
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
{search_result_json}
</QUESTION>
 """.strip()

#create LLM helper
#answer= llm(user_prompt,instructions)
answer=rag('how do i implement llm as judge?')
pprint(answer)

Loaded 95 documents
None
